# M³TM Tutorial: Building Your First AI App

Welcome to the interactive M³TM tutorial! In this hands-on guide, you'll learn how to:

- 🚀 Set up M³TM in your development environment
- 🧠 Load and initialize your first AI model
- 🖼️ Process text and images with multi-modal AI
- 🔍 Build a semantic search application
- 📱 Deploy to mobile devices

**Estimated time:** 30 minutes  
**Prerequisites:** Basic Python knowledge  
**What you'll build:** A photo search app that understands natural language queries

---

## Learning Objectives

By the end of this tutorial, you will:

✅ Understand M³TM's core concepts and architecture  
✅ Know how to process multi-modal data (text + images)  
✅ Be able to build semantic search functionality  
✅ Have a working app ready for mobile deployment  

Let's get started! 🎯

## Step 1: Environment Setup

First, let's install M³TM and verify everything is working correctly.

In [ ]:
# Install M³TM and dependencies
!pip install m3tm[tutorial] --quiet

# Verify installation
import m3tm
print(f"✅ M³TM v{m3tm.__version__} installed successfully!")
print(f"📦 Available backends: {m3tm.get_available_backends()}")
print(f"🔧 GPU available: {m3tm.cuda.is_available()}")

### Import Required Libraries

Let's import everything we'll need for this tutorial:

In [ ]:
import m3tm
from m3tm.config import ModelConfig, ModelSize, PrivacyMode
from m3tm.tasks import SimilaritySearch, SemanticSearch
from m3tm.data import TextInput, ImageInput, MultiModalInput

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO
import json
from pathlib import Path

# Configure matplotlib for notebook
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)

print("📚 All libraries imported successfully!")

## Step 2: Understanding M³TM Concepts

Before we dive into coding, let's understand the key concepts:

### 🧠 **Multi-Modal Processing**
M³TM can understand both text and images simultaneously, creating rich semantic representations.

### 🔐 **Privacy-First Design**
All processing happens on-device. Your data never leaves the user's control.

### ⚡ **Lightweight Architecture**
Optimized for mobile devices with minimal memory and CPU usage.

### 🎯 **Task-Specific Optimization**
Different tasks (search, classification, similarity) use specialized heads for best performance.

## Step 3: Initialize Your First Model

Let's create and configure an M³TM model:

In [ ]:
# Create model configuration
config = ModelConfig(
    model_size=ModelSize.COMPACT,  # Optimized for mobile
    privacy_mode=PrivacyMode.STRICT,  # Maximum privacy protection
    enable_caching=True,  # Cache results for faster repeated queries
    max_sequence_length=512,  # Maximum text length
    image_size=(224, 224),  # Optimal image resolution
    batch_size=1  # Process one item at a time
)

print("🔧 Configuration created:")
print(f"   Model size: {config.model_size.value}")
print(f"   Privacy mode: {config.privacy_mode.value}")
print(f"   Cache enabled: {config.enable_caching}")

In [ ]:
# Initialize the model
print("🚀 Initializing M³TM model...")

model = m3tm.M3TM(config=config)

# Warm up the model (loads weights and optimizes for inference)
print("🔥 Warming up model...")
model.warm_up()

print("✅ Model ready for inference!")
print(f"📊 Model info:")
print(f"   Parameters: {model.num_parameters:,}")
print(f"   Memory usage: {model.memory_usage_mb:.1f} MB")
print(f"   Supported tasks: {model.supported_tasks}")

## Step 4: Your First Multi-Modal Inference

Let's process some text and images to see M³TM in action!

In [ ]:
# Prepare sample data
sample_text = "A beautiful sunset over the mountains with golden light"

# Load a sample image (we'll create a simple one for the demo)
def create_sample_image():
    """Create a sample sunset image for the demo."""
    # Create a gradient image representing a sunset
    width, height = 224, 224
    image_array = np.zeros((height, width, 3), dtype=np.uint8)
    
    for y in range(height):
        # Create gradient from orange (top) to dark blue (bottom)
        ratio = y / height
        r = int(255 * (1 - ratio) + 50 * ratio)  # Orange to dark
        g = int(200 * (1 - ratio) + 30 * ratio)  # Orange to dark  
        b = int(50 * (1 - ratio) + 100 * ratio)  # Orange to blue
        
        image_array[y, :] = [r, g, b]
    
    return Image.fromarray(image_array)

sample_image = create_sample_image()

# Display the sample data
print(f"📝 Text: \"{sample_text}\"")
print(f"🖼️ Image: {sample_image.size} sunset gradient")

# Show the image
plt.figure(figsize=(6, 6))
plt.imshow(sample_image)
plt.title("Sample Image: Sunset Gradient")
plt.axis('off')
plt.show()

In [ ]:
# Create multi-modal input
multimodal_input = MultiModalInput(
    text=TextInput(sample_text),
    image=ImageInput(sample_image)
)

print("🔄 Running inference...")

# Run inference
result = model.infer(
    input_data=multimodal_input,
    task=SimilaritySearch()  # We want to measure text-image similarity
)

print("✅ Inference complete!")
print(f"\n📊 Results:")
print(f"   Similarity score: {result.similarity_score:.3f}")
print(f"   Text embedding shape: {result.text_embedding.shape}")
print(f"   Image embedding shape: {result.image_embedding.shape}")
print(f"   Fused embedding shape: {result.fused_embedding.shape}")
print(f"   Inference time: {result.inference_time_ms:.1f} ms")

### Understanding the Results

Let's interpret what we just computed:

In [ ]:
# Interpret similarity score
similarity = result.similarity_score

if similarity > 0.7:
    interpretation = "🟢 Strong match - Text and image are highly related"
elif similarity > 0.4:
    interpretation = "🟡 Moderate match - Some semantic similarity detected"
else:
    interpretation = "🔴 Weak match - Text and image don't seem related"

print(f"🎯 Similarity Interpretation:")
print(f"   Score: {similarity:.3f}")
print(f"   {interpretation}")

# Visualize embeddings
print(f"\n🧮 Embedding Analysis:")
print(f"   Text embedding magnitude: {np.linalg.norm(result.text_embedding):.3f}")
print(f"   Image embedding magnitude: {np.linalg.norm(result.image_embedding):.3f}")
print(f"   Cosine similarity: {np.dot(result.text_embedding, result.image_embedding) / (np.linalg.norm(result.text_embedding) * np.linalg.norm(result.image_embedding)):.3f}")

## Step 5: Building a Photo Search Application

Now let's build something practical - a photo search app that understands natural language queries!

In [ ]:
class PhotoSearchApp:
    """A simple photo search application using M³TM."""
    
    def __init__(self, model):
        self.model = model
        self.photo_database = []
        self.embeddings_cache = {}
    
    def add_photo(self, image, description, tags=None):
        """Add a photo to the searchable database."""
        photo_id = len(self.photo_database)
        
        # Generate embedding for the image
        image_input = ImageInput(image)
        result = self.model.infer(
            input_data=image_input,
            task=SemanticSearch()
        )
        
        photo_entry = {
            'id': photo_id,
            'image': image,
            'description': description,
            'tags': tags or [],
            'embedding': result.image_embedding,
            'added_date': pd.Timestamp.now() if 'pd' in globals() else 'now'
        }
        
        self.photo_database.append(photo_entry)
        print(f"📸 Added photo {photo_id}: {description}")
        
        return photo_id
    
    def search(self, query, top_k=5):
        """Search photos using natural language query."""
        if not self.photo_database:
            return []
        
        # Generate embedding for the search query
        text_input = TextInput(query)
        query_result = self.model.infer(
            input_data=text_input,
            task=SemanticSearch()
        )
        query_embedding = query_result.text_embedding
        
        # Calculate similarities with all photos
        similarities = []
        for photo in self.photo_database:
            # Use cosine similarity
            similarity = np.dot(query_embedding, photo['embedding']) / (
                np.linalg.norm(query_embedding) * np.linalg.norm(photo['embedding'])
            )
            similarities.append((photo['id'], similarity, photo))
        
        # Sort by similarity and return top results
        similarities.sort(key=lambda x: x[1], reverse=True)
        
        return similarities[:top_k]
    
    def display_search_results(self, query, results):
        """Display search results in a nice format."""
        print(f"🔍 Search results for: \"{query}\"")
        print(f"Found {len(results)} matching photos\n")
        
        if not results:
            print("No photos found. Try a different query.")
            return
        
        # Create subplot for results
        n_results = len(results)
        cols = min(3, n_results)
        rows = (n_results + cols - 1) // cols
        
        fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
        if rows == 1 and cols == 1:
            axes = [axes]
        elif rows == 1:
            axes = axes
        else:
            axes = axes.flatten()
        
        for i, (photo_id, similarity, photo) in enumerate(results):
            ax = axes[i] if n_results > 1 else axes[0]
            
            ax.imshow(photo['image'])
            ax.set_title(f"#{photo_id} ({similarity:.3f})\n{photo['description']}")
            ax.axis('off')
        
        # Hide unused subplots
        for i in range(n_results, len(axes)):
            axes[i].set_visible(False)
        
        plt.tight_layout()
        plt.show()

# Initialize the photo search app
photo_app = PhotoSearchApp(model)
print("📱 Photo Search App initialized!")

### Adding Sample Photos to Search Database

Let's create a diverse set of sample photos to search through:

In [ ]:
# Create diverse sample images for our photo database
def create_sample_photos():
    """Create a variety of sample photos with different themes."""
    photos = []
    
    # 1. Sunset scene
    sunset_img = create_sample_image()  # We already have this function
    photos.append((sunset_img, "Beautiful sunset over mountains", ["sunset", "mountains", "nature"]))
    
    # 2. Ocean scene (blue gradient)
    ocean_array = np.zeros((224, 224, 3), dtype=np.uint8)
    for y in range(224):
        ratio = y / 224
        ocean_array[y, :] = [int(100 * (1-ratio)), int(150 * (1-ratio) + 100 * ratio), 255]
    ocean_img = Image.fromarray(ocean_array)
    photos.append((ocean_img, "Calm blue ocean with clear sky", ["ocean", "water", "blue", "calm"]))
    
    # 3. Forest scene (green gradient)
    forest_array = np.zeros((224, 224, 3), dtype=np.uint8)
    for y in range(224):
        ratio = y / 224
        forest_array[y, :] = [int(50 * (1-ratio)), int(150 + 50 * ratio), int(50 * (1-ratio))]
    forest_img = Image.fromarray(forest_array)
    photos.append((forest_img, "Dense green forest with tall trees", ["forest", "trees", "green", "nature"]))
    
    # 4. Desert scene (sandy colors)
    desert_array = np.zeros((224, 224, 3), dtype=np.uint8)
    for y in range(224):
        ratio = y / 224
        desert_array[y, :] = [int(255 * (1-ratio) + 200 * ratio), int(200 * (1-ratio) + 150 * ratio), int(100 * (1-ratio) + 50 * ratio)]
    desert_img = Image.fromarray(desert_array)
    photos.append((desert_img, "Vast sandy desert with dunes", ["desert", "sand", "dunes", "dry"]))
    
    # 5. Night sky (dark with stars)
    night_array = np.zeros((224, 224, 3), dtype=np.uint8)
    night_array[:, :] = [20, 20, 50]  # Dark blue base
    # Add some "stars" (white dots)
    for _ in range(50):
        x, y = np.random.randint(0, 224, 2)
        night_array[y:y+2, x:x+2] = [255, 255, 255]
    night_img = Image.fromarray(night_array)
    photos.append((night_img, "Starry night sky with twinkling stars", ["night", "stars", "sky", "dark"]))
    
    return photos

# Create and add sample photos
sample_photos = create_sample_photos()

print("📸 Adding photos to database...")
for image, description, tags in sample_photos:
    photo_app.add_photo(image, description, tags)

print(f"\n✅ Added {len(sample_photos)} photos to searchable database!")

### Let's Search!

Now comes the fun part - let's search through our photos using natural language:

In [ ]:
# Search example 1: Looking for warm colors
query1 = "warm golden colors and peaceful scenery"
results1 = photo_app.search(query1, top_k=3)
photo_app.display_search_results(query1, results1)

In [ ]:
# Search example 2: Looking for nature scenes
query2 = "natural landscape with plants and trees"
results2 = photo_app.search(query2, top_k=3)
photo_app.display_search_results(query2, results2)

In [ ]:
# Search example 3: Looking for water
query3 = "blue water and aquatic environment"
results3 = photo_app.search(query3, top_k=3)
photo_app.display_search_results(query3, results3)

In [ ]:
# Interactive search - try your own query!
print("🎮 Try your own search query!")
print("Available photos contain: sunset, ocean, forest, desert, night sky")
print("\nExample queries to try:")
print("  - 'bright and sunny weather'")
print("  - 'dark mysterious atmosphere'")
print("  - 'colorful and vibrant scene'")
print("  - 'peaceful and calming environment'")

# Uncomment the next line to enable interactive input
# custom_query = input("\nEnter your search query: ")
# For demo purposes, let's use a predefined query
custom_query = "bright and colorful outdoor scene"

print(f"\n🔍 Searching for: '{custom_query}'")
custom_results = photo_app.search(custom_query, top_k=5)
photo_app.display_search_results(custom_query, custom_results)

## Step 6: Performance Analysis

Let's analyze the performance of our photo search app:

In [ ]:
import time

def benchmark_search_performance(app, queries, iterations=5):
    """Benchmark search performance across multiple queries."""
    
    results = {
        'query_times': [],
        'embedding_times': [],
        'similarity_times': [],
        'total_times': []
    }
    
    print(f"🏃 Running performance benchmark ({iterations} iterations)...")
    
    for query in queries:
        query_times = []
        
        for i in range(iterations):
            start_time = time.time()
            
            # Time the search operation
            search_results = app.search(query, top_k=3)
            
            end_time = time.time()
            query_time = (end_time - start_time) * 1000  # Convert to milliseconds
            query_times.append(query_time)
        
        avg_time = np.mean(query_times)
        std_time = np.std(query_times)
        
        results['total_times'].extend(query_times)
        
        print(f"   Query: '{query[:30]}...' - Avg: {avg_time:.1f}ms (±{std_time:.1f}ms)")
    
    # Overall statistics
    overall_avg = np.mean(results['total_times'])
    overall_std = np.std(results['total_times'])
    
    print(f"\n📊 Performance Summary:")
    print(f"   Average search time: {overall_avg:.1f} ms")
    print(f"   Standard deviation: {overall_std:.1f} ms")
    print(f"   Min time: {min(results['total_times']):.1f} ms")
    print(f"   Max time: {max(results['total_times']):.1f} ms")
    
    return results

# Test queries for benchmarking
benchmark_queries = [
    "sunset over mountains",
    "blue ocean water",
    "green forest trees",
    "sandy desert landscape",
    "starry night sky"
]

perf_results = benchmark_search_performance(photo_app, benchmark_queries, iterations=3)

In [ ]:
# Visualize performance results
plt.figure(figsize=(12, 5))

# Plot 1: Search time distribution
plt.subplot(1, 2, 1)
plt.hist(perf_results['total_times'], bins=10, alpha=0.7, color='skyblue', edgecolor='black')
plt.xlabel('Search Time (ms)')
plt.ylabel('Frequency')
plt.title('Search Time Distribution')
plt.axvline(np.mean(perf_results['total_times']), color='red', linestyle='--', 
           label=f'Mean: {np.mean(perf_results["total_times"]):.1f}ms')
plt.legend()

# Plot 2: Performance vs Database Size simulation
plt.subplot(1, 2, 2)
db_sizes = [1, 5, 10, 25, 50, 100]
# Simulate linear scaling with database size
base_time = np.mean(perf_results['total_times'])
simulated_times = [base_time * (1 + 0.1 * np.log(size)) for size in db_sizes]

plt.plot(db_sizes, simulated_times, 'bo-', linewidth=2, markersize=6)
plt.xlabel('Database Size (number of photos)')
plt.ylabel('Estimated Search Time (ms)')
plt.title('Scalability Projection')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 Performance Insights:")
print(f"   • Current database has {len(photo_app.photo_database)} photos")
print(f"   • Average search completes in {np.mean(perf_results['total_times']):.1f}ms")
print(f"   • Memory usage: ~{model.memory_usage_mb:.1f}MB")
print(f"   • Ready for mobile deployment! 📱")

## Step 7: Mobile Deployment Preparation

Now let's prepare our app for mobile deployment:

In [ ]:
# Export model for mobile deployment
def prepare_for_mobile(model, output_dir='./mobile_export'):
    """Prepare M³TM model for mobile deployment."""
    
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)
    
    print(f"📱 Preparing model for mobile deployment...")
    
    # Export optimized model
    mobile_model_path = output_path / "m3tm_mobile.pt"
    model.export_for_mobile(mobile_model_path)
    
    # Generate configuration file
    config_data = {
        "model_version": "2.3.0",
        "model_size": model.config.model_size.value,
        "input_image_size": model.config.image_size,
        "max_sequence_length": model.config.max_sequence_length,
        "supported_tasks": model.supported_tasks,
        "deployment_date": str(pd.Timestamp.now()) if 'pd' in globals() else "2024-01-01",
        "performance_metrics": {
            "avg_inference_time_ms": float(np.mean(perf_results['total_times'])),
            "memory_usage_mb": model.memory_usage_mb,
            "model_size_mb": mobile_model_path.stat().st_size / (1024*1024) if mobile_model_path.exists() else 4.2
        }
    }
    
    config_path = output_path / "model_config.json"
    with open(config_path, 'w') as f:
        json.dump(config_data, f, indent=2)
    
    # Generate sample integration code
    android_code = f"""
// Android Integration Example
class PhotoSearchActivity : AppCompatActivity() {{
    private lateinit var m3tmModel: M3TM
    
    override fun onCreate(savedInstanceState: Bundle?) {{
        super.onCreate(savedInstanceState)
        
        val config = ModelConfig.builder()
            .setModelSize(ModelSize.{model.config.model_size.value.upper()})
            .setPrivacyMode(PrivacyMode.{model.config.privacy_mode.value.upper()})
            .build()
            
        m3tmModel = M3TM.builder()
            .setConfig(config)
            .setModelPath("m3tm_mobile.pt")
            .build()
    }}
    
    fun searchPhotos(query: String): List<SearchResult> {{
        return m3tmModel.search(query, topK = 5)
    }}
}}
"""
    
    ios_code = f"""
// iOS Integration Example
class PhotoSearchViewController: UIViewController {{
    private var m3tmModel: M3TM?
    
    override func viewDidLoad() {{
        super.viewDidLoad()
        
        let config = ModelConfig(
            modelSize: .{model.config.model_size.value},
            privacyMode: .{model.config.privacy_mode.value}
        )
        
        do {{
            m3tmModel = try M3TM(config: config, modelPath: "m3tm_mobile.pt")
        }} catch {{
            print("Failed to initialize M³TM: \\(error)")
        }}
    }}
    
    func searchPhotos(query: String) -> [SearchResult] {{
        return m3tmModel?.search(query, topK: 5) ?? []
    }}
}}
"""
    
    # Save integration examples
    (output_path / "android_integration.kt").write_text(android_code)
    (output_path / "ios_integration.swift").write_text(ios_code)
    
    print(f"✅ Mobile deployment package ready!")
    print(f"   📂 Output directory: {output_path}")
    print(f"   📊 Model size: {config_data['performance_metrics']['model_size_mb']:.1f} MB")
    print(f"   ⚡ Avg inference: {config_data['performance_metrics']['avg_inference_time_ms']:.1f} ms")
    print(f"   🧠 Memory usage: {config_data['performance_metrics']['memory_usage_mb']:.1f} MB")
    
    return output_path

# Prepare for mobile deployment
mobile_package = prepare_for_mobile(model)

## 🎉 Congratulations!

You've successfully built your first AI-powered photo search application with M³TM! Here's what you accomplished:

### ✅ What You Built

- **🧠 Multi-modal AI model** that understands text and images
- **🔍 Semantic search engine** with natural language queries
- **📱 Mobile-ready deployment** package
- **⚡ Performance-optimized** application

### 🔑 Key Concepts Learned

1. **Privacy-first AI**: All processing happens on-device
2. **Multi-modal understanding**: Combining text and image analysis
3. **Semantic similarity**: Finding meaning beyond keyword matching
4. **Mobile optimization**: Efficient inference for mobile devices

### 🚀 Next Steps

Ready to take your skills further? Here are some ideas:

1. **Expand your dataset**: Add more diverse photos and test different queries
2. **Add new features**: Implement image classification or content generation
3. **Optimize further**: Experiment with different model sizes and configurations
4. **Deploy to mobile**: Use the generated integration code in a real app
5. **Build something new**: Create a note-taking app, document search, or personal assistant

### 📚 Additional Resources

- [M³TM Documentation](https://docs.m3tm.dev) - Complete API reference
- [Example Gallery](https://examples.m3tm.dev) - More sample applications
- [Community Forum](https://community.m3tm.dev) - Get help and share projects
- [GitHub Repository](https://github.com/m3tm/mobilemodel) - Source code and issues

---

**Happy building! 🛠️**

*Remember: With great AI power comes great responsibility. Always respect user privacy and build inclusive, beneficial applications.*

In [ ]:
# Final summary and next steps
print("🎓 Tutorial Complete! Summary:")
print(f"   • Processed {len(photo_app.photo_database)} photos")
print(f"   • Average search time: {np.mean(perf_results['total_times']):.1f}ms")
print(f"   • Model memory usage: {model.memory_usage_mb:.1f}MB")
print(f"   • Ready for mobile deployment: ✅")
print(f"\n📦 Your mobile package is ready at: {mobile_package}")
print(f"\n🔗 Next tutorials to explore:")
print(f"   • Advanced Performance Optimization")
print(f"   • Custom Model Training")
print(f"   • Production Deployment Guide")
print(f"   • Building a Complete Mobile App")

# Display final demo
print(f"\n🌟 Final Demo - Search for 'peaceful nature scene':")
final_results = photo_app.search("peaceful nature scene", top_k=3)
photo_app.display_search_results("peaceful nature scene", final_results)